# 01 — Logging professionnel

## Objectifs pédagogiques

À la fin de ce notebook, vous saurez :

- configurer le module `logging` stdlib
- comprendre loggers, handlers, formatters et la hiérarchie
- utiliser `dictConfig` pour une configuration avancée
- connaître `loguru` et `structlog` en tant qu'alternatives

## Prérequis — ce que vous connaissez déjà

À ce stade de la formation intermédiaire, vous maîtrisez :

- fichiers, modules, décorateurs, context managers

Ce que nous n'avons **pas encore vu** (et que nous n'utiliserons donc pas dans ce notebook) :

- `structlog` approfondi (formation Avancé)

## Plan

1. Pourquoi logger (pas `print`)
2. Premier usage : `logging.basicConfig`
3. Loggers hiérarchiques
4. Handlers et Formatters
5. `dictConfig`
6. Alternatives : `loguru`, `structlog`
7. Synthèse
8. Exercices

---

## 1. Pourquoi logger (pas `print`)

`print` est pratique en dev mais inadapté en production :

- pas de niveau de gravité (debug, info, warning, error)
- pas de redirection vers fichier/syslog/monitoring
- pas de filtrage par module
- mélange avec la sortie standard du programme

---

## 2. Premier usage : `basicConfig`

In [ ]:
import logging

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s - %(name)s - %(message)s')


In [ ]:
logger = logging.getLogger('mon_app')
logger.debug('message de debug')
logger.info('informations')
logger.warning('attention')
logger.error('erreur')


---

## 3. Loggers hiérarchiques

Le nom du logger est hiérarchique (séparateur `.`). `mon_app.db` est enfant de `mon_app`.

In [ ]:
db_logger = logging.getLogger('mon_app.db')
db_logger.info('connexion établie')


Les messages remontent au parent. Un handler sur `'mon_app'` recevra aussi les messages de `'mon_app.db'`.

---

## 4. Handlers et Formatters

Un handler détermine **où** le message va (console, fichier, réseau). Un formatter détermine **comment** il est formaté.

In [ ]:
import logging

logger = logging.getLogger('demo')
logger.setLevel(logging.DEBUG)

# Handler console
ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
ch.setFormatter(logging.Formatter('%(asctime)s [%(levelname)s] %(message)s'))
logger.addHandler(ch)

logger.info('visible sur la console')
logger.debug('pas visible (niveau trop bas pour le handler)')


---

## 5. `dictConfig`

Configuration structurée recommandée en production.

In [ ]:
import logging.config

config = {
    'version': 1,
    'disable_existing_loggers': False,
    'formatters': {
        'standard': {'format': '%(asctime)s [%(levelname)s] %(name)s: %(message)s'},
    },
    'handlers': {
        'console': {
            'class': 'logging.StreamHandler',
            'formatter': 'standard',
            'level': 'INFO',
        },
    },
    'root': {
        'handlers': ['console'],
        'level': 'DEBUG',
    },
}

logging.config.dictConfig(config)
logging.getLogger().info('configuré via dictConfig')


---

## 6. Alternatives : `loguru`, `structlog`

La stdlib `logging` est puissante mais verbeuse. Deux alternatives populaires :

### `loguru`

```python
from loguru import logger
logger.info('message')
```

Zero config, couleurs, rotation automatique, sérialisation JSON.

### `structlog`

```python
import structlog
log = structlog.get_logger()
log.info('message', user='Alice', action='login')
```

Logging structuré (JSON, key-value), idéal pour les pipelines d'observabilité.

---

## Synthèse

| Concept | Rôle |
|---|---|
| Logger | Point d'entrée du message |
| Handler | Où va le message (console, fichier…) |
| Formatter | Mise en forme |
| Level | DEBUG < INFO < WARNING < ERROR < CRITICAL |
| `dictConfig` | Configuration structurée |


### Règles à retenir

1. **`logging.getLogger(__name__)`** dans chaque module.
2. **`basicConfig` pour le prototypage**, `dictConfig` en production.
3. **Ne loggez jamais de données sensibles** (mots de passe, tokens).
4. **Utilisez `logger.exception(...)` dans un bloc `except`** pour inclure le traceback.

---

## Exercices

Les exercices sont gradués. Tous utilisent des fonctions typées (PEP 604).

### Exercice 1 — Logger basique *(facile)*

Configurer un logger pour votre module et écrire un message à chaque niveau.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Logging_pro", exercice=1)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging

logging.basicConfig(level=logging.DEBUG, format='%(levelname)s: %(message)s')
log = logging.getLogger('exo')
log.debug('debug'); log.info('info'); log.warning('warn'); log.error('err')
```

</details>

### Exercice 2 — Logger avec handler fichier *(moyen)*

Créer un logger avec deux handlers : console (INFO) et fichier temporaire (DEBUG). Écrire un message DEBUG et un INFO, vérifier que seul l'INFO apparaît en console.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Logging_pro", exercice=2)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging, tempfile

log = logging.getLogger('dual')
log.setLevel(logging.DEBUG)

ch = logging.StreamHandler()
ch.setLevel(logging.INFO)
log.addHandler(ch)

f = tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False)
fh = logging.FileHandler(f.name)
fh.setLevel(logging.DEBUG)
log.addHandler(fh)

log.debug('debug seulement dans le fichier')
log.info('info sur console ET fichier')
print(f'fichier : {f.name}')
```

</details>

### Exercice 3 — Décorateur `@log_call` *(difficile)*

Écrire un décorateur `@log_call(logger)` qui log à INFO le nom de la fonction, les arguments et le retour.

In [ ]:
# Votre code ici


In [ ]:
# ▶ Une fois votre solution écrite ci-dessus, exécutez cette cellule
# pour signaler à votre formateur que vous avez tenté l'exercice.
import sys
from pathlib import Path
for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "_common" / "utils_pedagogie.py").exists():
        sys.path.insert(0, str(_p / "_common")); break
from utils_pedagogie import marquer_tentative
marquer_tentative(notebook="01_Logging_pro", exercice=3)


<details>
<summary>📖 Voir la correction</summary>

```python
import logging, functools
from typing import Any
from collections.abc import Callable

def log_call(log: logging.Logger) -> Callable:
    def deco(func: Callable) -> Callable:
        @functools.wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            log.info(f'APPEL {func.__name__}(args={args}, kwargs={kwargs})')
            result = func(*args, **kwargs)
            log.info(f'RETOUR {func.__name__} → {result}')
            return result
        return wrapper
    return deco

logging.basicConfig(level=logging.INFO, format='%(message)s')
log = logging.getLogger('exo')

@log_call(log)
def carre(x: int) -> int:
    return x * x

carre(5)
```

</details>

---

## Ressources externes

### Documentation officielle
- [`logging` — stdlib](https://docs.python.org/3/library/logging.html)
- [Logging HOWTO](https://docs.python.org/3/howto/logging.html)
- [loguru](https://github.com/Delgan/loguru)
- [structlog](https://www.structlog.org/)